In [1]:
import sqlite3
import pandas as pd

from pathlib import Path

In [4]:
try:
    # Localização Rigorosa da Base de Dados
    # Procura o arquivo na pasta atual e vai subindo até encontrar
    current_path = Path(".").resolve()
    db_path = None
    
    # Este loop procura o banco na pasta atual e nos diretórios pais
    for folder in [current_path] + list(current_path.parents):
        candidate = folder / "aposta_ganha_dw.db"
        if candidate.exists():
            db_path = candidate
            break
            
    if not db_path:
        raise FileNotFoundError("Banco 'aposta_ganha_dw.db' não encontrado na raiz ou subpastas!")

    print(f"Conectando ao banco oficial em: {db_path}")

    # Conexão Segura
    conn = sqlite3.connect(str(db_path))
    cursor = conn.cursor()
    
    # Diagnóstico: listar tabelas para garantir que estamos no banco certo
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tabelas_existentes = [t[0] for t in cursor.fetchall()]
    print(f"Tabelas encontradas no banco: {tabelas_existentes}")

    # Validação da tabela alvo
    if 'tabela_final_modelada' not in tabelas_existentes:
        raise Exception("A tabela 'tabela_final_modelada' não existe neste arquivo! Verifique se rodou o train_clustering.ipynb até o fim.")

    # Integração
    print("Extraindo dados...")
    df = pd.read_sql_query("SELECT * FROM tabela_final_modelada;", conn)
    
    # Exportação para CSV na raiz do projeto
    output_path = db_path.parent / "analytics_outputs.csv"
    df.to_csv(output_path, index=False)
    
    print(f"Integração finalizada com sucesso!")
    print(f"Arquivo 'analytics_outputs.csv' exportado para: {output_path}")
    
    conn.close()

except Exception as e:
    print(f"Falha na integração: {e}")

Conectando ao banco oficial em: C:\Users\ewert\Documents\Projetos\aposta-ganha-analytics-framework\aposta_ganha_dw.db
Tabelas encontradas no banco: ['dim_usuarios', 'fato_transacoes', 'tabela_analytics', 'tabela_final_modelada']
Extraindo dados...
Integração finalizada com sucesso!
Arquivo 'analytics_outputs.csv' exportado para: C:\Users\ewert\Documents\Projetos\aposta-ganha-analytics-framework\analytics_outputs.csv
